# Phase 6c Wave 2: EW Baryogenesis ↔ Chirality Wall — Technical Notebook

Companion to `papers/paper33_ewbg_chirality_wall/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/EWBaryogenesisChiralityWall.lean` (16 substantive theorems / 0 sorry / 0 new axioms).

**Structure:** Two-pillar bridge of (a) chirality-wall obstruction (Phase 5e/5p Z₁₆ anomaly classification) × (b) EW phase-transition order (Phase 5z.3) → SM EWBG verdict.

**Punchline:** SM-as-is is **doubly forbidden** under H_KLRS — two independent failure modes (wall intact for SM-no-ν_R; transition crossover for SM+3ν_R).

## 1. Setup

Import the chirality-wall predicates, sphaleron helpers, and SM verdict dataclass from `src.ew_baryogenesis`.

In [1]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.core.constants import EW_PARAMS, EWBG_PARAMS
from src.core.formulas import (
    sphaleron_suppression,
    chirality_wall_blocks_ewbg,
    ewbg_viable,
)
from src.ew_baryogenesis import (
    sm_ewbg_verdict,
    sm_no_nu_R_z16_anomaly,
    sm_with_3nu_R_z16_anomaly,
    sphaleron_decoupling_threshold,
    sphalerons_decoupled,
    wall_intact,
    wall_cracked,
)

print(f"SM m_H               = {EW_PARAMS['M_H_GEV']} GeV (PDG 2024)")
print(f"KLRS endpoint        = {EWBG_PARAMS['KLRS_M_H_CROSSOVER_THRESHOLD_GEV']} GeV (CFH 1999)")
print(f"Sphaleron threshold  = {sphaleron_decoupling_threshold()} (Cohen-Kaplan-Nelson 1993)")
print(f"SM-no-ν_R Z₁₆        = {sm_no_nu_R_z16_anomaly()} (= 3 × 15 mod 16)")
print(f"SM+3ν_R Z₁₆          = {sm_with_3nu_R_z16_anomaly()} (= 3 × 16 mod 16)")
print(f"M_H overshoot ratio  = {EWBG_PARAMS['M_H_OVERSHOOT_RATIO']:.3f}")

SM m_H               = 125.2 GeV (PDG 2024)
KLRS endpoint        = 72.4 GeV (CFH 1999)
Sphaleron threshold  = 1.0 (Cohen-Kaplan-Nelson 1993)
SM-no-ν_R Z₁₆        = 13 (= 3 × 15 mod 16)
SM+3ν_R Z₁₆          = 0 (= 3 × 16 mod 16)
M_H overshoot ratio  = 1.729


## 2. Sphaleron suppression structural form

The dimensionless suppression factor $\Sigma(v, T) = e^{-v/T}$ embeds the Klinkhamer-Manton sphaleron rate's exponential suppression in the broken phase. Cohen-Kaplan-Nelson (1993) decoupling: $v(T_c)/T_c > 1$.

**Lean:** `sphaleronSuppression_pos`, `sphaleronSuppression_le_one`, `SphaleronsDecoupled`.

In [2]:
for v_over_T in [0.0, 0.5, 1.0, 1.5, 2.0, 5.0]:
    T = 100.0  # GeV
    v = v_over_T * T
    sigma = sphaleron_suppression(v, T)
    decoupled = sphalerons_decoupled(v, T)
    print(f"v/T = {v_over_T:.1f}  →  Σ = {sigma:.4e}, decoupled = {decoupled}")

v/T = 0.0  →  Σ = 1.0000e+00, decoupled = False
v/T = 0.5  →  Σ = 6.0653e-01, decoupled = False
v/T = 1.0  →  Σ = 3.6788e-01, decoupled = False
v/T = 1.5  →  Σ = 2.2313e-01, decoupled = True
v/T = 2.0  →  Σ = 1.3534e-01, decoupled = True
v/T = 5.0  →  Σ = 6.7379e-03, decoupled = True


## 3. Chirality-wall predicate over Z₁₆

`WallIntact(a) := a ≠ 0 (mod 16)`; the gauging step requires anomaly cancellation (Phase 5p `gauging_requires_z16_cancellation`).

**Substantive cross-bridges to `Z16AnomalyComputation` (Phase 5e):**
- `sm_no_nu_R_wall_intact` consumes `three_gen_anomalous` (45 ≠ 0 in Z₁₆)
- `sm_single_gen_with_nu_R_wall_cracks` consumes `sm_anomaly_with_nu_R` (16 ≡ 0 in Z₁₆)
- `sm_with_3nu_R_wall_cracks` extends to 3 generations (48 ≡ 0)

In [3]:
print("Z₁₆ chirality-wall predicate over candidate fermion contents:")
print(f"  SM-no-ν_R:  anomaly = {sm_no_nu_R_z16_anomaly()},  wall intact = {wall_intact(sm_no_nu_R_z16_anomaly())}")
print(f"  SM+3ν_R:    anomaly = {sm_with_3nu_R_z16_anomaly()},  wall intact = {wall_intact(sm_with_3nu_R_z16_anomaly())}")
print()
print("Canonical-representative bridge: 45 mod 16 = 13 (= -3 mod 16)")
print(f"  45 mod 16 = {45 % 16}  ({chirality_wall_blocks_ewbg(45) == chirality_wall_blocks_ewbg(13)})")

Z₁₆ chirality-wall predicate over candidate fermion contents:
  SM-no-ν_R:  anomaly = 13,  wall intact = True
  SM+3ν_R:    anomaly = 0,  wall intact = False

Canonical-representative bridge: 45 mod 16 = 13 (= -3 mod 16)
  45 mod 16 = 13  (True)


## 4. The compound `EWBGViable` predicate

`EWBGViable(a, p, c) := WallCracked(a) ∧ IsBaryogenesisViable(p, c)`. EWBG fails if EITHER condition fails.

**Lean:** `EWBGViable`, `ewbg_forbidden_if_wall_intact`, `ewbg_forbidden_if_transition_crossover`. Both consume substantive upstream theorems in proof bodies.

**Substantive correctness-push biconditional:** `ewbg_forbidden_iff_wall_intact_or_not_viable`. Every breakdown of EWBG must be attributed to either the wall or the phase transition; no third failure mode in this two-pillar decomposition.

In [4]:
# Three regimes of (anomaly, transition_strength)
SM_BENCH = dict(mu_sq=7744.0, lam=0.13, c_T=0.4)

scenarios = [
    ("SM-no-ν_R, strict-LO E=0.01",          13, 0.01),
    ("SM+3ν_R, strict-LO E=0.01",             0, 0.01),
    ("SM+3ν_R, full-thermal crossover E=0",   0, 0.0),
    ("SM+3ν_R, BSM strong first-order E=25",  0, 25.0),
    ("SM-no-ν_R, BSM strong first-order E=25", 13, 25.0),
]
for label, anomaly, E in scenarios:
    viable = ewbg_viable(anomaly, E=E, **SM_BENCH)
    print(f"  {label:40s}  →  EWBGViable = {viable}")

  SM-no-ν_R, strict-LO E=0.01               →  EWBGViable = False
  SM+3ν_R, strict-LO E=0.01                 →  EWBGViable = False
  SM+3ν_R, full-thermal crossover E=0       →  EWBGViable = False
  SM+3ν_R, BSM strong first-order E=25      →  EWBGViable = True
  SM-no-ν_R, BSM strong first-order E=25    →  EWBGViable = False


## 5. SM verdict — the doubly-forbidden punchline

**Tracked Prop:** `H_KLRS_SM_Crossover` encodes the lattice result that the resummed-thermal SM transition is a crossover for $m_H > 72.4$ GeV. SM has $m_H = 125.20$ GeV, an overshoot ratio $> 1.5$.

**Lean:** `sm_klrs_overshoot_ratio_gt_threshold` (1.5 < 125.20/72.4 via norm_num), `sm_no_nu_R_ewbg_doubly_forbidden`.

In [5]:
verdict = sm_ewbg_verdict(sm_full_is_crossover=True)
print("=" * 60)
print("SM EWBG verdict under H_KLRS:")
print("=" * 60)
print(f"  KLRS overshoot ratio        : {verdict.klrs_overshoot_ratio:.3f}× (> 1.5)")
print(f"  SM-no-ν_R wall intact?      : {verdict.sm_no_nu_R_wall_intact}")
print(f"  SM-no-ν_R EWBG blocked?     : {verdict.sm_no_nu_R_ewbg_blocked}")
print(f"  SM+3ν_R wall cracks?        : {verdict.sm_with_3nu_R_wall_cracks}")
print(f"  SM+3ν_R EWBG under H_KLRS?  : {verdict.sm_with_3nu_R_ewbg_under_klrs}")
print()
print(f"  DOUBLY FORBIDDEN?           : {verdict.doubly_forbidden()}")
assert verdict.doubly_forbidden(), "SM EWBG must be doubly forbidden"
print("\n  ✓ Punchline confirmed: SM-as-is fails BOTH conditions")

SM EWBG verdict under H_KLRS:
  KLRS overshoot ratio        : 1.729× (> 1.5)
  SM-no-ν_R wall intact?      : True
  SM-no-ν_R EWBG blocked?     : True
  SM+3ν_R wall cracks?        : True
  SM+3ν_R EWBG under H_KLRS?  : False

  DOUBLY FORBIDDEN?           : True

  ✓ Punchline confirmed: SM-as-is fails BOTH conditions


## 6. Figure: EWBG verdict over (wall × transition)

**viz-ref:** `fig_ewbg_allowed_region`

In [6]:
from src.core.visualizations import fig_ewbg_allowed_region
fig = fig_ewbg_allowed_region()
fig.show()

## 7. Cross-references and next steps

**Cross-bridges (Lean proof bodies):**
- Phase 5e `Z16AnomalyComputation`: `three_gen_anomalous`, `sm_anomaly_with_nu_R` — chirality-wall integrity
- Phase 5p `GaugingStep`: `gauging_requires_z16_cancellation` — gauging step requires anomaly cancellation
- Phase 5a Wave 4 `ChiralityWallMaster`: three-pillar synthesis
- Phase 5z.3 `EWPhaseTransition`: `crossover_excludes_baryogenesis`, `IsBaryogenesisViable`

**Dispatches to:**
- **Leptogenesis** via Phase 5z Wave 2 `MajoranaRung.lean` sterile-neutrino seesaw (M_R ~ 10¹⁴ GeV) — out-of-equilibrium decays generate L asymmetry, converted to B by sphalerons above EWPT
- **BSM EWBG** via extra-scalar models (singlet, 2HDM, MSSM with light stops) producing strong first-order EWPT
- **Phase 6e** (nonlinear stress-energy) inherits the verdict for stress-energy structure decisions

**Tracked-Prop validation status:** `H_KLRS_SM_Crossover` is a project-external hypothesis (lattice result, not derived in our framework). Discharging it requires lattice-thermodynamics infrastructure (Phase 6b.3 Boltzmann-code nucleation rates is the natural candidate).